# 2D Gaussian Mixture Langevin

Chapter 18: Langevin Dynamics and Sampling · *Mathematics for AI and Machine Learning* by Fuheng Wu

Score-based Langevin sampling on a two-mode 2D density.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-18/mixture-2d/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import torch

torch.manual_seed(142)

# The demo's target: equal mixture of N((2, 0), 0.5 I) and N((-2, 0), 0.5 I)
m = torch.tensor([[2.0, 0.0], [-2.0, 0.0]], dtype=torch.float64)
var = 0.5
steps, burn_in, eta = 2000, 500, 0.01


def score(x):
    w = torch.softmax(-((x[:, None, :] - m) ** 2).sum(-1) / (2 * var), dim=1)
    return (w[:, :, None] * (-(x[:, None, :] - m) / var)).sum(1)


# ── One chain, as in the demo: it stays in the mode it falls into ───────────
x = torch.zeros(1, 2, dtype=torch.float64)
trace = []
for _ in range(steps):
    x = x + eta * score(x) + (2 * eta) ** 0.5 * torch.randn_like(x)
    trace.append(x[0])
trace = torch.stack(trace[burn_in:])
print(f"single chain: fraction with x > 0 = {(trace[:, 0] > 0).double().mean().item():.3f} (target 0.5)")
print("-> modes 4 apart with variance 0.5: crossing is very rare, so one chain sees one mode")

# ── Many chains from the same start split evenly between the modes ──────────
chains = torch.zeros(4000, 2, dtype=torch.float64)
for _ in range(steps):
    chains = chains + eta * score(chains) + (2 * eta) ** 0.5 * torch.randn_like(chains)
right = (chains[:, 0] > 0).double().mean().item()
print(f"\n4000 chains: fraction in the right mode = {right:.3f}")
cov_right = torch.cov(chains[chains[:, 0] > 0].T)
print("covariance within the right mode =", [[round(v, 3) for v in r] for r in cov_right.tolist()], "(target 0.5 I)")

single chain: fraction with x > 0 = 0.000 (target 0.5)
-> modes 4 apart with variance 0.5: crossing is very rare, so one chain sees one mode



4000 chains: fraction in the right mode = 0.509
covariance within the right mode = [[0.491, -0.0], [-0.0, 0.51]] (target 0.5 I)
